# DL 시퀀스 데이터 준비
Purpose: build deterministic causal sequence indexes from the verified Goal 1.5 ML role views.

> Warning: this is an oracle/sanity-only synthetic-data benchmark, not real-device or medical-performance evidence.

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

SERIES_ID = "mvp3-oracle-v1"
EXPECTED_SPLIT_COUNTS = {"train": 24, "validation": 6, "locked_test": 6}
DATA_STATUS = "oracle/sanity"
REAL_ACCURACY_STATUS = "NOT VERIFIED"
DEVICE_SYNCHRONIZATION_STATUS = "NOT_AVAILABLE_TRUTH_ONLY"
RUN_TRAINING = False
RUN_LOCKED_TEST = False
SEQUENCE_LENGTHS_SECONDS = (300, 600)
SEQUENCE_OUTPUT_ROOT = Path("/kaggle/working/goal15_dl_sequences")
ML_VIEW_ROOT = Path("/kaggle/working/goal15_ml_view")
PATTERN_TARGET = "pattern_binary"
ONSET_EVENT_TARGET = "event_binary"
STAGE_TARGET = "stage_code"
STAGE_CODES = ("NO_EVENT", "LOW", "MEDIUM", "HIGH", "DECREASING", "RECOVERY")
BEHAVIOR_CODES = (
    "ear_covering", "exit_attempt", "head_turn_away", "motion_freeze",
    "movement_reduction", "repetitive_body_movement",
    "repetitive_hand_movement", "repetitive_object_contact",
    "sustained_pressure_or_contact", "withdrawal_movement",
)
CAUSAL_FACTORS = (
    "autonomic_arousal", "motor_activation", "cognitive_load", "sleep_pressure",
    "sensory_context", "recovery_capacity", "social_context",
)
ROLLING_STATISTICS = ("mean", "std", "slope")
ROLLING_WINDOWS_SECONDS = (5, 15, 30, 60, 180, 300)
TIME_FEATURE_COLUMNS = ("time_sin", "time_cos", "weekday_sin", "weekday_cos", "is_awake")
CONTEXT_FEATURE_COLUMNS = (
    "context__sleep", "context__transition", "context__meal_context",
    "context__focused_task", "context__moderate_activity",
    "context__light_activity", "context__wake_rest",
    "context__sedentary_activity",
)
ALLOWED_FEATURE_COLUMNS = tuple(
    [
        feature
        for factor in CAUSAL_FACTORS
        for feature in (
            f"{factor}__robust_z",
            *(
                f"{factor}__{statistic}_{window_seconds}s"
                for window_seconds in ROLLING_WINDOWS_SECONDS
                for statistic in ROLLING_STATISTICS
            ),
        )
    ]
    + list(TIME_FEATURE_COLUMNS)
    + list(CONTEXT_FEATURE_COLUMNS)
)
SEQUENCE_IDENTITY_COLUMNS = ("person_key", "run_id", "dataset_id", "canonical_time", "context")
SEQUENCE_LABEL_COLUMNS = (PATTERN_TARGET, ONSET_EVENT_TARGET, "hard_negative", STAGE_TARGET, *BEHAVIOR_CODES)
SEQUENCE_METADATA_COLUMNS = frozenset({
    *SEQUENCE_IDENTITY_COLUMNS, "person_id", "split_role", "event_id", "session_id",
    "day_key", "day", "date", "missing_block", "is_missing_block",
    *SEQUENCE_LABEL_COLUMNS,
})


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _require_sha256(value: Any, field: str) -> str:
    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:
        raise ValueError(f"invalid {field} hash")
    return value


def derive_flat_dataset_identity(dataset_root: Path) -> tuple[str, str]:
    manifest_names = ("prepared__manifest.json", "outcomes__manifest.json", "registry__manifest.json")
    manifest_hashes: dict[str, str] = {}
    for name in manifest_names:
        path = dataset_root / name
        if not path.is_file():
            raise FileNotFoundError(f"missing original Dataset manifest: {name}")
        json.loads(path.read_text())
        manifest_hashes[name] = sha256_file(path)
    split_path = dataset_root / "registry__splits.parquet"
    if not split_path.is_file():
        raise FileNotFoundError("missing original Dataset split registry")
    source_dataset_hash = hashlib.sha256(json.dumps(manifest_hashes, sort_keys=True).encode()).hexdigest()
    return source_dataset_hash, sha256_file(split_path)


def validate_shared_dataset_identity(
    manifest: Mapping[str, Any], dataset_root: Path
) -> tuple[str, str]:
    if manifest.get("series_id") != SERIES_ID:
        raise ValueError("unexpected series identity")
    if manifest.get("data_status") != DATA_STATUS:
        raise ValueError("unexpected data status")
    declared_source = _require_sha256(manifest.get("source_dataset_hash"), "source dataset")
    declared_split = _require_sha256(manifest.get("split_hash"), "split")
    source_dataset_hash, split_hash = derive_flat_dataset_identity(dataset_root)
    if declared_source != source_dataset_hash:
        raise ValueError("source dataset hash does not match original flat Dataset")
    if declared_split != split_hash:
        raise ValueError("split hash does not match original flat Dataset")
    return source_dataset_hash, split_hash


def fit_train_normalization(
    frame: pd.DataFrame, *, feature_columns: Sequence[str], source_hash: str
) -> dict[str, Any]:
    _require_sha256(source_hash, "source")
    if "split_role" not in frame:
        raise ValueError("normalization requires split_role")
    missing = sorted(set(feature_columns).difference(frame.columns))
    if missing:
        raise ValueError(f"normalization features missing: {missing}")
    train = frame.loc[frame["split_role"].eq("train"), list(feature_columns)]
    if train.empty:
        raise ValueError("normalization requires train rows")
    statistics: dict[str, dict[str, float]] = {}
    for feature in feature_columns:
        values = pd.to_numeric(train[feature], errors="raise").to_numpy(dtype=np.float64)
        if not np.isfinite(values).all():
            raise ValueError(f"non-finite train feature: {feature}")
        median = float(np.median(values))
        lower, upper = np.quantile(values, [0.25, 0.75])
        iqr = float(upper - lower)
        statistics[feature] = {"median": median, "iqr": iqr if iqr > 0 else 1.0}
    return {
        "series_id": SERIES_ID,
        "fit_split_role": "train",
        "source_hash": source_hash,
        "features": statistics,
    }


def _require_non_null(frame: pd.DataFrame, columns: Sequence[str]) -> None:
    missing = sorted(set(columns).difference(frame.columns))
    if missing:
        raise ValueError(f"sequence contract missing columns: {missing}")
    null_columns = [column for column in columns if frame[column].isna().any()]
    if null_columns:
        raise ValueError(f"sequence contract has null columns: {null_columns}")


def _require_binary(frame: pd.DataFrame, columns: Sequence[str]) -> None:
    _require_non_null(frame, columns)
    invalid = [column for column in columns if not frame[column].isin((0, 1)).all()]
    if invalid:
        raise ValueError(f"sequence contract requires binary labels: {invalid}")


def validate_sequence_role_frame(frame: pd.DataFrame, split_role: str) -> list[str]:
    if split_role not in EXPECTED_SPLIT_COUNTS:
        raise ValueError(f"unknown split role: {split_role}")
    _require_non_null(frame, SEQUENCE_IDENTITY_COLUMNS)
    _require_binary(frame, (PATTERN_TARGET, ONSET_EVENT_TARGET, "hard_negative", *BEHAVIOR_CODES))
    if frame[STAGE_TARGET].isna().any() or not frame[STAGE_TARGET].isin(STAGE_CODES).all():
        raise ValueError("sequence contract has invalid stage_code")
    expected_pattern = frame[STAGE_TARGET].ne("NO_EVENT").astype("int8")
    if not frame[PATTERN_TARGET].astype("int8").eq(expected_pattern).all():
        raise ValueError("pattern_binary must equal stage_code != NO_EVENT")
    if (frame[PATTERN_TARGET].eq(1) & frame["hard_negative"].eq(1)).any():
        raise ValueError("pattern and hard_negative cannot overlap")
    behavior_positive = frame.loc[:, list(BEHAVIOR_CODES)].eq(1).any(axis=1)
    ordinary_behavior = behavior_positive & frame[PATTERN_TARGET].eq(0) & frame["hard_negative"].eq(0)
    if ordinary_behavior.any():
        raise ValueError("behavior-positive ordinary baseline is not allowed")
    missing_features = [feature for feature in ALLOWED_FEATURE_COLUMNS if feature not in frame]
    if missing_features:
        raise ValueError(f"missing approved features: {missing_features}")
    unexpected = sorted(set(frame.columns).difference(ALLOWED_FEATURE_COLUMNS).difference(SEQUENCE_METADATA_COLUMNS))
    if unexpected:
        raise ValueError(f"unapproved DL sequence columns: {unexpected}")
    for feature in ALLOWED_FEATURE_COLUMNS:
        if not (pd.api.types.is_numeric_dtype(frame[feature]) or pd.api.types.is_bool_dtype(frame[feature])):
            raise ValueError(f"approved feature must be numeric or bool: {feature}")
        if not np.isfinite(frame[feature].to_numpy(dtype=np.float64)).all():
            raise ValueError(f"non-finite approved feature: {feature}")
    return list(ALLOWED_FEATURE_COLUMNS)


def _window_group_keys(frame: pd.DataFrame) -> list[str]:
    required = [*SEQUENCE_IDENTITY_COLUMNS, "split_role", PATTERN_TARGET, "hard_negative"]
    missing = sorted(set(required).difference(frame.columns))
    if missing:
        raise ValueError(f"sequence source missing columns: {missing}")
    group_keys = ["person_key", "run_id", "dataset_id", "context"]
    group_keys.extend(key for key in ("day_key", "day", "date", "session_id") if key in frame)
    return group_keys


def assert_window_boundaries(index: pd.DataFrame) -> None:
    required = {"person_key", "run_id", "dataset_id", "window_start", "window_end", "prediction_time", "length_seconds"}
    missing = sorted(required.difference(index.columns))
    if missing:
        raise ValueError(f"sequence index missing columns: {missing}")
    # Every candidate must satisfy window_end >= window_start.
    if not (index["window_end"] >= index["window_start"]).all():
        raise ValueError("window_end must be after window_start")
    expected_end = index["prediction_time"]
    if not index["window_end"].eq(expected_end).all():
        raise ValueError("window_end must equal prediction_time")
    expected_start = expected_end - pd.to_timedelta(index["length_seconds"] - 1, unit="s")
    if not index["window_start"].eq(expected_start).all():
        raise ValueError("causal window start mismatch")


In [ ]:
def make_causal_window_index(frame: pd.DataFrame, *, length_seconds: int) -> pd.DataFrame:
    if isinstance(length_seconds, bool) or length_seconds < 1:
        raise ValueError("length_seconds must be positive")
    group_keys = _window_group_keys(frame)
    work = frame.copy()
    _require_non_null(work, ["person_key", "run_id", "dataset_id", "context", "canonical_time", "split_role"])
    _require_binary(work, (PATTERN_TARGET, "hard_negative"))
    work["canonical_time"] = pd.to_datetime(work["canonical_time"], utc=True)
    if work["canonical_time"].isna().any():
        raise ValueError("canonical_time must be complete")
    missing_column = next((name for name in ("missing_block", "is_missing_block") if name in work), None)
    records: list[dict[str, Any]] = []
    for _, group in work.groupby(group_keys, sort=False, dropna=False):
        ordered = group.sort_values("canonical_time", kind="mergesort").reset_index(drop=True)
        if ordered["canonical_time"].duplicated().any():
            raise ValueError("duplicate canonical_time within sequence group")
        for end_index in range(length_seconds - 1, len(ordered)):
            window = ordered.iloc[end_index - length_seconds + 1 : end_index + 1]
            prediction_time = window["canonical_time"].iloc[-1]
            window_start = prediction_time - pd.Timedelta(seconds=length_seconds - 1)
            window_end = prediction_time
            consecutive = window["canonical_time"].diff().dropna().eq(pd.Timedelta(seconds=1)).all()
            has_missing_block = bool(window[missing_column].fillna(True).astype(bool).any()) if missing_column else False
            if not consecutive or has_missing_block:
                continue
            record = window.iloc[-1].to_dict()
            record.update({
                "window_start": window_start,
                "window_end": window_end,
                "prediction_time": prediction_time,
                "length_seconds": length_seconds,
            })
            record["window_id"] = hashlib.sha256(
                "|".join(str(record[key]) for key in (*group_keys, "prediction_time", "length_seconds")).encode()
            ).hexdigest()
            records.append(record)
    columns = [*frame.columns, "window_start", "window_end", "prediction_time", "length_seconds", "window_id"]
    index = pd.DataFrame(records, columns=list(dict.fromkeys(columns)))
    if not index.empty:
        assert_window_boundaries(index)
    return index


def sample_training_windows(index: pd.DataFrame, *, baseline_multiplier: int = 3) -> pd.DataFrame:
    if baseline_multiplier < 0:
        raise ValueError("baseline_multiplier must be non-negative")
    required = {"person_key", "run_id", "dataset_id", "context", "split_role", PATTERN_TARGET, "hard_negative", "length_seconds", "prediction_time", "window_id"}
    missing = sorted(required.difference(index.columns))
    if missing:
        raise ValueError(f"training index missing columns: {missing}")
    if not index["split_role"].eq("train").all():
        raise ValueError("training sampler accepts train windows only")
    _require_non_null(index, ["person_key", "run_id", "dataset_id", "context", "length_seconds", "prediction_time", "window_id"])
    _require_binary(index, (PATTERN_TARGET, "hard_negative"))
    positive = index[PATTERN_TARGET].eq(1)
    hard_negative = ~positive & index["hard_negative"].eq(1)
    baseline = ~(positive | hard_negative)
    selected = [
        index.loc[positive].assign(sample_type="positive_centered"),
        index.loc[hard_negative].assign(sample_type="hard_negative"),
    ]
    strata = ["person_key", "run_id", "context", "length_seconds"]
    baseline_rows: list[pd.DataFrame] = []
    for _, positives in index.loc[positive].groupby(strata, sort=False, dropna=False):
        key = tuple(positives.iloc[0][column] for column in strata)
        matched = index.loc[baseline].copy()
        for column, value in zip(strata, key, strict=True):
            matched = matched.loc[matched[column].eq(value)]
        matched["_sample_hash"] = pd.util.hash_pandas_object(
            matched[["person_key", "run_id", "dataset_id", "context", "prediction_time", "window_id"]],
            index=False, categorize=True,
        )
        baseline_rows.append(
            matched.sort_values(["_sample_hash", "window_id"], kind="mergesort")
            .head(baseline_multiplier * len(positives))
            .drop(columns="_sample_hash")
            .assign(sample_type="matched_baseline")
        )
    selected.extend(baseline_rows)
    return pd.concat(selected, ignore_index=True).sort_values(
        ["prediction_time", "window_id"], kind="mergesort"
    ).reset_index(drop=True)


def write_train_normalization(output_root: Path, statistics: Mapping[str, Any]) -> Path:
    output_root.mkdir(parents=True, exist_ok=True)
    path = output_root / "train_normalization.json"
    path.write_text(json.dumps(statistics, indent=2, sort_keys=True) + "\n")
    return path


def write_sequence_manifest(
    output_root: Path, *, index_paths: Mapping[str, Path], normalization_path: Path,
    source_dataset_hash: str, split_hash: str, row_counts: Mapping[str, int],
) -> Path:
    _require_sha256(source_dataset_hash, "source dataset")
    _require_sha256(split_hash, "split")
    if set(index_paths) != set(row_counts):
        raise ValueError("sequence manifest role metadata mismatch")
    if not normalization_path.is_file():
        raise FileNotFoundError("missing train normalization statistics")
    files: dict[str, dict[str, Any]] = {}
    for name, path in sorted(index_paths.items()):
        count = row_counts[name]
        if not path.is_file() or isinstance(count, bool) or not isinstance(count, int) or count < 0:
            raise ValueError(f"invalid sequence output: {name}")
        files[name] = {"path": path.name, "sha256": sha256_file(path), "row_count": count}
    manifest_path = output_root / "sequence_manifest.json"
    manifest_path.write_text(json.dumps({
        "series_id": SERIES_ID, "data_status": DATA_STATUS,
        "source_dataset_hash": source_dataset_hash, "split_hash": split_hash,
        "normalization": {"path": normalization_path.name, "sha256": sha256_file(normalization_path)},
        "files": files,
    }, indent=2, sort_keys=True) + "\n")
    return manifest_path


In [ ]:
def resolve_ml_view_root(input_root: Path = Path("/kaggle/input")) -> Path:
    children = sorted(path for path in input_root.iterdir() if path.is_dir()) if input_root.exists() else []
    for candidate in [ML_VIEW_ROOT, input_root, *children]:
        if (candidate / "view_manifest.json").is_file():
            return candidate
    raise FileNotFoundError("verified goal15_ml_view manifest is required")


def resolve_flat_dataset_root(input_root: Path = Path("/kaggle/input")) -> Path:
    children = sorted(path for path in input_root.iterdir() if path.is_dir()) if input_root.exists() else []
    required = {"prepared__manifest.json", "outcomes__manifest.json", "registry__manifest.json", "registry__splits.parquet"}
    for candidate in [input_root, *children]:
        if required.issubset({path.name for path in candidate.iterdir()}):
            return candidate
    raise FileNotFoundError("original flat Goal 1.5 Dataset is required")


def _flat_split_membership(dataset_root: Path) -> dict[str, set[str]]:
    split = pq.read_table(dataset_root / "registry__splits.parquet", columns=["person_key", "split_role"]).to_pandas()
    _require_non_null(split, ["person_key", "split_role"])
    if not split["split_role"].isin(EXPECTED_SPLIT_COUNTS).all():
        raise ValueError("original split has invalid role")
    memberships = {role: set(split.loc[split["split_role"].eq(role), "person_key"]) for role in EXPECTED_SPLIT_COUNTS}
    if {role: len(memberships[role]) for role in EXPECTED_SPLIT_COUNTS} != EXPECTED_SPLIT_COUNTS:
        raise ValueError("original split is not 24/6/6")
    if any(memberships[left] & memberships[right] for left in memberships for right in memberships if left < right):
        raise ValueError("original split has person overlap")
    return memberships


def _verified_role_views(root: Path, dataset_root: Path) -> tuple[dict[str, Path], str, str]:
    manifest = json.loads((root / "view_manifest.json").read_text())
    source_dataset_hash, split_hash = validate_shared_dataset_identity(manifest, dataset_root)
    files = manifest.get("files")
    if not isinstance(files, dict) or set(files) != set(EXPECTED_SPLIT_COUNTS):
        raise ValueError("ML view manifest must declare all immutable split roles")
    expected_membership = _flat_split_membership(dataset_root)
    paths: dict[str, Path] = {}
    expected_feature_types: dict[str, pa.DataType] | None = None
    for split_role, metadata in files.items():
        path = root / metadata["path"]
        if not path.is_file() or sha256_file(path) != _require_sha256(metadata.get("sha256"), split_role):
            raise ValueError(f"unverified ML role view: {split_role}")
        parquet = pq.ParquetFile(path)
        columns = list(parquet.schema_arrow.names)
        if metadata.get("columns") != columns:
            raise ValueError(f"ML role view schema metadata mismatch: {split_role}")
        missing_features = [feature for feature in ALLOWED_FEATURE_COLUMNS if feature not in columns]
        if missing_features:
            raise ValueError(f"missing approved features: {missing_features}")
        feature_types = {feature: parquet.schema_arrow.field(feature).type for feature in ALLOWED_FEATURE_COLUMNS}
        if expected_feature_types is None:
            expected_feature_types = feature_types
        elif feature_types != expected_feature_types:
            raise ValueError(f"feature schema mismatch: {split_role}")
        people: set[str] = set()
        for row_group in range(parquet.num_row_groups):
            person_table = parquet.read_row_group(row_group, columns=["person_key"])
            person_values = person_table.column("person_key").to_pandas()
            if person_values.isna().any():
                raise ValueError("ML role view has null person_key")
            people.update(person_values.tolist())
        if people != expected_membership[split_role]:
            raise ValueError(f"ML role view membership mismatch: {split_role}")
        paths[split_role] = path
    return paths, source_dataset_hash, split_hash


def _iter_role_person_chunks(path: Path, split_role: str):
    parquet = pq.ParquetFile(path)
    for row_group in range(parquet.num_row_groups):
        frame = parquet.read_row_group(row_group).to_pandas()
        if "split_role" not in frame:
            frame["split_role"] = split_role
        elif not frame["split_role"].eq(split_role).all():
            raise ValueError(f"role column mismatch: {split_role}")
        if frame["person_key"].nunique(dropna=False) != 1:
            raise ValueError("ML role row group must be person-bounded")
        validate_sequence_role_frame(frame, split_role)
        yield frame


def fit_train_normalization_from_role_file(
    path: Path, *, feature_columns: Sequence[str], source_hash: str
) -> dict[str, Any]:
    _require_sha256(source_hash, "source")
    parquet = pq.ParquetFile(path)
    statistics: dict[str, dict[str, float]] = {}
    for feature in feature_columns:
        chunks: list[np.ndarray] = []
        for row_group in range(parquet.num_row_groups):
            values = parquet.read_row_group(row_group, columns=[feature]).column(feature).to_numpy(zero_copy_only=False)
            numeric = np.asarray(values, dtype=np.float64)
            if not np.isfinite(numeric).all():
                raise ValueError(f"non-finite train feature: {feature}")
            chunks.append(numeric)
        values = np.concatenate(chunks) if chunks else np.array([], dtype=np.float64)
        if not len(values):
            raise ValueError(f"normalization has no train values: {feature}")
        lower, upper = np.quantile(values, [0.25, 0.75])
        iqr = float(upper - lower)
        statistics[feature] = {"median": float(np.median(values)), "iqr": iqr if iqr > 0 else 1.0}
    return {"series_id": SERIES_ID, "fit_split_role": "train", "source_hash": source_hash, "features": statistics}


def build_all_sequence_indexes(
    *, flat_dataset_root: Path | None = None, ml_view_root: Path | None = None, output_root: Path = SEQUENCE_OUTPUT_ROOT
) -> Path:
    dataset_root = flat_dataset_root or resolve_flat_dataset_root()
    views_root = ml_view_root or resolve_ml_view_root()
    role_paths, source_dataset_hash, split_hash = _verified_role_views(views_root, dataset_root)
    for split_role, path in role_paths.items():
        for _ in _iter_role_person_chunks(path, split_role):
            pass
    statistics = fit_train_normalization_from_role_file(
        role_paths["train"], feature_columns=ALLOWED_FEATURE_COLUMNS, source_hash=source_dataset_hash
    )
    normalization_path = write_train_normalization(output_root, statistics)
    output_root.mkdir(parents=True, exist_ok=True)
    index_paths: dict[str, Path] = {}
    row_counts: dict[str, int] = {}
    writers: dict[str, pq.ParquetWriter] = {}
    schemas: dict[str, pa.Schema] = {}
    try:
        for split_role, path in role_paths.items():
            for frame in _iter_role_person_chunks(path, split_role):
                for length_seconds in SEQUENCE_LENGTHS_SECONDS:
                    index = make_causal_window_index(frame, length_seconds=length_seconds)
                    if split_role == "train":
                        index = sample_training_windows(index)
                    name = f"{split_role}_{length_seconds}"
                    table = pa.Table.from_pandas(index, preserve_index=False)
                    if name not in writers:
                        index_path = output_root / f"{name}.parquet"
                        writers[name] = pq.ParquetWriter(index_path, table.schema, compression="zstd")
                        schemas[name] = table.schema
                        index_paths[name] = index_path
                        row_counts[name] = 0
                    elif not table.schema.equals(schemas[name], check_metadata=False):
                        raise ValueError(f"inconsistent sequence index schema: {name}")
                    writers[name].write_table(table)
                    row_counts[name] += len(index)
    finally:
        for writer in writers.values():
            writer.close()
    expected_names = {f"{role}_{length}" for role in EXPECTED_SPLIT_COUNTS for length in SEQUENCE_LENGTHS_SECONDS}
    if set(index_paths) != expected_names:
        raise ValueError("missing sequence index output")
    return write_sequence_manifest(
        output_root, index_paths=index_paths, normalization_path=normalization_path,
        source_dataset_hash=source_dataset_hash, split_hash=split_hash, row_counts=row_counts,
    )


In [ ]:
RUN_DATA_PREPARATION = False

if RUN_DATA_PREPARATION:
    manifest_path = build_all_sequence_indexes()
    print(f"시퀀스 인덱스 생성 완료: {manifest_path}")
else:
    print("시퀀스 데이터 준비 비활성화: RUN_DATA_PREPARATION=False")
